In [1]:
import json
import pandas as pd
import numpy as np
import os

# 1. Load and Inspect Bangalore Ward GeoJSON

This cell loads the `bangalore_wards.json` file and inspects its structure.

It checks:

- whether the file is a dictionary or list
- the top-level keys
- whether the file is in GeoJSON format
- the total number of features
- sample ward properties from the first few entries

This helps confirm that the ward boundary/mapping file has been loaded correctly before further preprocessing.

In [3]:
with open('bangalore_wards.json') as f:
    data = json.load(f)
    
print(f"Type: {type(data)}")
if isinstance(data, dict):
    print(f"Keys: {list(data.keys())}")
elif isinstance(data, list):
    print(f"Length: {len(data)}")
    print(f"First item keys: {list(data[0].keys()) if isinstance(data[0], dict) else data[0]}")

# Check if it's GeoJSON
if isinstance(data, dict) and 'type' in data:
    print(f"\nGeoJSON type: {data['type']}")
    if 'features' in data:
        print(f"Number of features: {len(data['features'])}")
        f0 = data['features'][0]
        print(f"Feature keys: {list(f0.keys())}")
        print(f"Properties: {f0['properties']}")
        print(f"\nFirst 5 features:")
        for feat in data['features'][:5]:
            print(f" {feat['properties']}")

Type: <class 'dict'>
Keys: ['type', 'features']

GeoJSON type: FeatureCollection
Number of features: 198
Feature keys: ['type', 'properties', 'geometry']
Properties: {'WARD_NO': '2', 'WARD_NAME': 'Chowdeswari Ward', 'MOVEMENT_ID': '1', 'DISPLAY_NAME': 'Unnamed Road, Bengaluru'}

First 5 features:
 {'WARD_NO': '2', 'WARD_NAME': 'Chowdeswari Ward', 'MOVEMENT_ID': '1', 'DISPLAY_NAME': 'Unnamed Road, Bengaluru'}
 {'WARD_NO': '3', 'WARD_NAME': 'Atturu', 'MOVEMENT_ID': '2', 'DISPLAY_NAME': '9th Cross Bhel Layout, Adityanagar, Vidyaranyapura, Bengaluru'}
 {'WARD_NO': '4', 'WARD_NAME': 'Yelahanka Satellite Town', 'MOVEMENT_ID': '3', 'DISPLAY_NAME': '15th A Cross Road, Yelahanka Satellite Town, Yelahanka, Bengaluru'}
 {'WARD_NO': '51', 'WARD_NAME': 'Vijnanapura', 'MOVEMENT_ID': '4', 'DISPLAY_NAME': 'SP Naidu Layout 4th Cross Street, SP Naidu Layout, Dooravani Nagar, Bengaluru'}
 {'WARD_NO': '53', 'WARD_NAME': 'Basavanapura', 'MOVEMENT_ID': '5', 'DISPLAY_NAME': 'Medahalli Kadugodi Road, Bharathi

# 2. Build Ward Mapping Table

This cell extracts relevant metadata from the GeoJSON features and builds a structured mapping table.

The generated dataframe `df_map` contains:

- `movement_id`
- `ward_no`
- `ward_name`
- `display_name`

The table is sorted by `movement_id`.

In [4]:
# Build full mapping table
rows = []
for feat in data['features']:
    p = feat['properties']
    rows.append({
        'movement_id': int(p['MOVEMENT_ID']),
        'ward_no': p['WARD_NO'],
        'ward_name': p['WARD_NAME'],
        'display_name': p['DISPLAY_NAME']
    })
df_map = pd.DataFrame(rows).sort_values('movement_id')
df_map

,movement_id,ward_no,ward_name,display_name
0,1,2,Chowdeswari Ward,"Unnamed Road, Bengaluru"
1,2,3,Atturu,"9th Cross Bhel Layout, Adityanagar, Vidyaranya..."
2,3,4,Yelahanka Satellite Town,"15th A Cross Road, Yelahanka Satellite Town, Y..."
3,4,51,Vijnanapura,"SP Naidu Layout 4th Cross Street, SP Naidu Lay..."
4,5,53,Basavanapura,"Medahalli Kadugodi Road, Bharathi Nagar, Krish..."
...,...,...,...,...
193,194,172,Madivala,"0 1st B Cross Road, Cashier Layout, 1st Stage,..."
194,195,26,Ramamurthy Nagar,"Kalkere-Agara Main Road, Horamavu Agara, Kalke..."
195,196,25,Horamavu,"0 Horamavu Agara Main Road, 1st Block, Mallapp..."
196,197,86,Marathahalli,"0 3rd Cross Road, Manjunatha Layout, Marathaha..."


# 3. Map Selected Bangalore Zones to Representative Ward IDs

This cell defines the mapping between the selected Bangalore study zones and their corresponding ward-based `movement_id` values.

The workflow includes:

- assigning one representative `movement_id` to each zone
- loading all four quarterly hourly aggregate traffic files
- combining them into a single dataframe
- filtering the data to retain only trips between the selected 16 zones
- removing self-loop trips where source and destination are the same
- adding readable source and destination zone names
- merging inter-zone distance values from the OSRM distance matrix
- saving the filtered dataset as `uber_bangalore_2018_processed.csv`

In [6]:
# Our 15 zones + depot → best matching ward IDs
zone_mapping = {
    'Whitefield': [167, 168, 169, 170],  # Garudachar Playa, Kadugodi, Hagadur, Dodda Nekkundi
    'Koramangala': [157],  # Koramangala
    'Indiranagar': [80],  # Hoysala Nagar (Indiranagar)
    'Hebbal': [55, 56],  # Hebbala, Vishwanath Nagenahalli
    'Marathahalli': [197],  # Marathahalli
    'Electronic City': [172, 175],  # Bommanahalli area
    'Jayanagar': [140, 149, 150],  # Jayanagar, Karisandra, Yediyur
    'Rajajinagar': [109],  # Rajaji Nagar
    'Yeshwanthpur': [25],  # Yeshwanthpura
    'BTM Layout': [161, 190],  # BTM Layout, Gurappanapalya
    'HSR Layout': [171],  # HSR Layout
    'Bannerghatta Rd': [179, 183],  # Arakere, Gottigere
    'Yelahanka': [3, 186],  # Yelahanka Satellite Town, Kempegowda Ward
    'Sarjapur Road': [165, 166],  # Varthuru, Bellanduru
    'MG Road': [89, 94],  # Shantala Nagar, Gandhinagar
    'Banashankari': [144, 145, 146, 151],  # Girinagar, Katriguppe, Vidyapeeta, Banashankari Temple
}

# Use the representative (first) movement_id for each zone
primary_ids = {zone: ids[0] for zone, ids in zone_mapping.items()}

print("--- ZONE → MOVEMENT ID MAPPING ---")

for zone, mid in primary_ids.items():
    ward = df_map[df_map['movement_id'] == mid]['ward_name'].values[0]
    print(f" {zone:<22} → movement_id: {mid:>3} ({ward})")

# Load all 4 quarters and filter
print("\n--- LOADING ALL 4 QUARTERS ---")

dfs = []

for q in [1, 2, 3, 4]:
    path = f'bangalore-wards-2018-{q}-All-HourlyAggregate.csv'
    df_q = pd.read_csv(path)
    df_q['quarter'] = q
    dfs.append(df_q)

    print(f" Q{q}: {len(df_q):>7,} rows loaded")

df_all = pd.concat(dfs, ignore_index=True)

print(f"\n TOTAL: {len(df_all):,} rows combined")

# Filter to our 16 zone IDs
selected_ids = list(primary_ids.values())

df_filtered = df_all[
    df_all['sourceid'].isin(selected_ids) &
    df_all['dstid'].isin(selected_ids) &
    (df_all['sourceid'] != df_all['dstid'])
].copy()

print(f"\n--- AFTER FILTERING TO 16 ZONES ---")
print(f" Rows: {len(df_filtered):,}")

print(
    f" Unique OD pairs: "
    f"{df_filtered.groupby(['sourceid', 'dstid']).ngroups}"
)

print(f" Expected OD pairs: {16 * 15} (16 zones × 15 directions)")

# Add zone names
id_to_zone = {v: k for k, v in primary_ids.items()}

df_filtered['source_name'] = df_filtered['sourceid'].map(id_to_zone)
df_filtered['dest_name'] = df_filtered['dstid'].map(id_to_zone)

# Add distance from real matrix
dist_df = pd.read_csv('distance_matrix_osrm.csv', index_col=0)


def get_dist(row):
    try:
        return dist_df.loc[row['source_name'], row['dest_name']]
    except:
        return np.nan


df_filtered['distance_km'] = df_filtered.apply(get_dist, axis=1)

# Save
df_filtered.to_csv('uber_bangalore_2018_processed.csv', index=False)

print(
    f"\n Saved: uber_bangalore_2018_processed.csv "
    f"({len(df_filtered):,} rows)"
)

--- ZONE → MOVEMENT ID MAPPING ---
 Whitefield             → movement_id: 167 (Garudachar Playa)
 Koramangala            → movement_id: 157 (Koramangala)
 Indiranagar            → movement_id:  80 (Hoysala Nagar)
 Hebbal                 → movement_id:  55 (Hebbala)
 Marathahalli           → movement_id: 197 (Marathahalli)
 Electronic City        → movement_id: 172 (Bommanahalli)
 Jayanagar              → movement_id: 140 (Jayanagar)
 Rajajinagar            → movement_id: 109 (Rajaji Nagar)
 Yeshwanthpur           → movement_id:  25 (Yeshwanthpura)
 BTM Layout             → movement_id: 161 (BTM Layout)
 HSR Layout             → movement_id: 171 (HSR Layout)
 Bannerghatta Rd        → movement_id: 179 (Arakere)
 Yelahanka              → movement_id:   3 (Yelahanka Satellite Town)
 Sarjapur Road          → movement_id: 165 (Varthuru)
 MG Road                → movement_id:  89 (Shantala Nagar)
 Banashankari           → movement_id: 144 (Girinagar)

--- LOADING ALL 4 QUARTERS ---
 Q1: 824,9

In [7]:
df_filtered.reset_index(inplace=True,drop=True)
df_filtered

,sourceid,dstid,hod,mean_travel_time,standard_deviation_travel_time,geometric_mean_travel_time,geometric_standard_deviation_travel_time,quarter,source_name,dest_name,distance_km
0,55,157,0,1363.10,255.38,1343.56,1.18,1,Hebbal,Koramangala,15.69
1,25,55,9,1301.06,416.42,1243.27,1.34,1,Yeshwanthpur,Hebbal,9.79
2,80,109,20,2973.61,731.30,2896.02,1.25,1,Indiranagar,Rajajinagar,11.60
3,144,157,18,3207.95,898.67,3096.49,1.30,1,Banashankari,Koramangala,12.01
4,140,197,18,3530.07,1012.10,3391.62,1.33,1,Jayanagar,Marathahalli,17.23
...,...,...,...,...,...,...,...,...,...,...,...
22618,197,179,3,1593.91,249.73,1575.67,1.16,4,Marathahalli,Bannerghatta Rd,17.97
22619,197,179,8,2777.05,1086.17,2595.35,1.43,4,Marathahalli,Bannerghatta Rd,17.97
22620,197,179,13,2570.38,539.38,2524.84,1.20,4,Marathahalli,Bannerghatta Rd,17.97
22621,197,179,18,4128.97,1261.27,3939.59,1.36,4,Marathahalli,Bannerghatta Rd,17.97


# 4. Define Zone List and Zone Categories

This cell defines:

- the ordered list of selected Bangalore zones
- the functional category of each zone

The categories include:

- IT
- Commercial
- Residential
- Industrial

In [8]:
ZONES = [
    "Whitefield",
    "Marathahalli",
    "Yeshwanthpur",
    "Yelahanka",
    "Koramangala",
    "Indiranagar",
    "Hebbal",
    "Electronic City",
    "Jayanagar",
    "Rajajinagar",
    "BTM Layout",
    "HSR Layout",
    "Bannerghatta Rd",
    "Sarjapur Road",
    "MG Road",
    "Banashankari",
]

ZONE_TYPE = {
    "Whitefield": "IT",
    "Koramangala": "Commercial",
    "Indiranagar": "Commercial",
    "Hebbal": "Residential",
    "Marathahalli": "IT",
    "Electronic City": "IT",
    "Jayanagar": "Residential",
    "Yeshwanthpur": "Industrial",
    "BTM Layout": "Residential",
    "Rajajinagar": "Residential",
    "HSR Layout": "Commercial",
    "Bannerghatta Rd": "Residential",
    "Yelahanka": "Residential",
    "Sarjapur Road": "IT",
    "MG Road": "Commercial",
    "Banashankari": "Residential",
}

# 5. Final Preprocessing and Feature Engineering

The preprocessing steps include:

- loading the filtered travel data
- removing self-loop records
- converting mean and standard deviation travel times from seconds to minutes
- filtering out unreliable observations using the coefficient of variation (CV)
- removing very short-duration trips
- assigning source and destination zone categories
- generating one-hot encoded zone-type features
- creating temporal features such as morning peak, evening peak, off-peak, cyclic hour encoding, and monsoon indicators
- computing a speed-based proxy using distance and travel time
- encoding source and destination zones as numeric IDs
- selecting the final set of relevant features
- saving the cleaned dataset as `data_processed.csv`

The resulting dataframe is the final structured dataset used for analysis and predictive modeling.

In [9]:
def preprocess():
    df = pd.read_csv("uber_bangalore_2018_processed.csv")
    print(f" Rows loaded: {len(df):,}")

    # Remove self-loops
    df = df[df["source_name"] != df["dest_name"]].copy()

    # Convert to minutes
    df["mean_travel_time_min"] = (
        df["mean_travel_time"] / 60
    ).round(2)

    df["std_travel_time_min"] = (
        df["standard_deviation_travel_time"] / 60
    ).round(2)

    # Remove unreliable records (CV > 1.0)
    df["cv"] = df["std_travel_time_min"] / df["mean_travel_time_min"]

    before = len(df)
    df = df[df["cv"] <= 1.0].copy()

    print(f" Removed {before - len(df)} high-CV records")

    # Remove very short records
    df = df[df["mean_travel_time_min"] >= 1.0].copy()

    # Zone type features
    df["zone_type_src"] = df["source_name"].map(ZONE_TYPE)
    df["zone_type_dst"] = df["dest_name"].map(ZONE_TYPE)

    for zt in ["IT", "Commercial", "Residential", "Industrial"]:
        df[f"src_{zt.lower()}"] = (
            df["zone_type_src"] == zt
        ).astype(int)

        df[f"dst_{zt.lower()}"] = (
            df["zone_type_dst"] == zt
        ).astype(int)

    # Feature Engineering (Temporal features)
    df["is_peak_morning"] = df["hod"].isin(
        [7, 8, 9, 10]
    ).astype(int)

    df["is_peak_evening"] = df["hod"].isin(
        [17, 18, 19, 20]
    ).astype(int)

    df["is_off_peak"] = (
        ~df["hod"].isin([7, 8, 9, 10, 17, 18, 19, 20])
    ).astype(int)

    df["hod_sin"] = np.sin(
        2 * np.pi * df["hod"] / 24
    ).round(6)

    df["hod_cos"] = np.cos(
        2 * np.pi * df["hod"] / 24
    ).round(6)

    df["is_monsoon"] = df["quarter"].isin(
        [2, 3]
    ).astype(int)

    df["speed_proxy"] = (
        df["distance_km"] /
        (df["mean_travel_time_min"] / 60)
    ).round(4)

    # Numeric zone IDs
    zone_to_id = {z: i for i, z in enumerate(ZONES)}

    df["source_id"] = df["source_name"].map(zone_to_id)
    df["dest_id"] = df["dest_name"].map(zone_to_id)

    # Select and save final columns
    feature_cols = [
        "source_id",
        "source_name",
        "dest_id",
        "dest_name",
        "distance_km",
        "mean_travel_time_min",
        "std_travel_time_min",
        "hod",
        "is_peak_morning",
        "is_peak_evening",
        "is_off_peak",
        "is_monsoon",
        "quarter",
        "speed_proxy",
        "src_it",
        "dst_it",
        "src_commercial",
        "src_residential",
        "src_industrial",
        "dst_commercial",
        "dst_residential",
        "dst_industrial",
        "hod_sin",
        "hod_cos",
    ]

    df_final = df[feature_cols].dropna().copy()

    df_final.to_csv("data_processed.csv", index=False)

    return df_final


preprocess()

 Rows loaded: 22,623
 Removed 44 high-CV records


,source_id,source_name,dest_id,dest_name,distance_km,mean_travel_time_min,std_travel_time_min,hod,is_peak_morning,is_peak_evening,...,src_it,dst_it,src_commercial,src_residential,src_industrial,dst_commercial,dst_residential,dst_industrial,hod_sin,hod_cos
0,6,Hebbal,4,Koramangala,15.69,22.72,4.26,0,0,0,...,0,0,0,1,0,1,0,0,0.000000,1.000000
1,2,Yeshwanthpur,6,Hebbal,9.79,21.68,6.94,9,1,0,...,0,0,0,0,1,0,1,0,0.707107,-0.707107
2,5,Indiranagar,9,Rajajinagar,11.60,49.56,12.19,20,0,1,...,0,0,1,0,0,0,1,0,-0.866025,0.500000
3,15,Banashankari,4,Koramangala,12.01,53.47,14.98,18,0,1,...,0,0,0,1,0,1,0,0,-1.000000,-0.000000
4,8,Jayanagar,1,Marathahalli,17.23,58.83,16.87,18,0,1,...,0,1,0,1,0,0,0,0,-1.000000,-0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22618,1,Marathahalli,12,Bannerghatta Rd,17.97,26.57,4.16,3,0,0,...,1,0,0,0,0,0,1,0,0.707107,0.707107
22619,1,Marathahalli,12,Bannerghatta Rd,17.97,46.28,18.10,8,1,0,...,1,0,0,0,0,0,1,0,0.866025,-0.500000
22620,1,Marathahalli,12,Bannerghatta Rd,17.97,42.84,8.99,13,0,0,...,1,0,0,0,0,0,1,0,-0.258819,-0.965926
22621,1,Marathahalli,12,Bannerghatta Rd,17.97,68.82,21.02,18,0,1,...,1,0,0,0,0,0,1,0,-1.000000,-0.000000
